# Energy criticality

This notebook checks the results of the N-1 criticality analysis of the `inferred-provided-<region>` network, which records, for each component, the unmet demand when that component alone fails. Two rules in `workflow/2-losses/energy.smk` write the results: `build_energy_criticality_flow` (`src/energy/criticality.py`) fails each substation, transmission line and power plant and solves a power flow, and `build_energy_criticality_connectivity` (`src/energy/connectivity.py`) fails the same components and each distribution line, from the connectivity of the network alone. The last section covers the distribution lines; the others use the power-flow results. The notebook reads and plots the results and builds nothing.

## Method

The analysis sets the demand of each substation, models the transmission network and runs one power flow for each failed component.

1. **Demand by substation area.** Each substation supplies its service area: the nodes of the distribution network nearer to it along the distribution lines than to any other substation. An area's share of its island's demand is 0.6 times its share of the island's night-light radiance plus 0.4 times its share of the population: PyPSA-Earth's weighting of GDP and population, with night lights in place of GDP. The share multiplies the island's peak or average demand from the utility's annual report, and the result is placed on the substation's bus. The power flow does not include the distribution network; the last map draws it only to show the areas that lose supply. An island without transmission lines is modelled as a single bus, whose unmet demand is split between its areas in proportion to their demand.
2. **Transmission network.** The transmission lines, those above `max_radial_line_voltage_kv` (printed below; 35 kV by default), connect the substations, and each plant is attached to its substation. Lines at or below that voltage are distribution lines, left out of the power flow. A line's rating is its number of circuits times the thermal rating of one circuit of a standard PyPSA line type at the voltage the line runs at, the type chosen by the line's design voltage. A line circuits table, if the energy config names one (`criticality.line_circuits`), gives each line's design voltage, circuits and route; without it, each run of segments between two substations or junctions is one line, with one circuit designed for the voltage it runs at.
3. **Failure of each component.** Starting each time from the intact network, the analysis removes one component: a substation together with its plants and its area's demand, a transmission line with all its segments, or a single plant. A linear optimal power flow (PyPSA) then supplies as much demand as the remaining network allows; the demand it cannot supply is the unmet demand, demand minus supply.
4. **Causes of unmet demand.** Demand is unmet when (a) its own substation fails; (b) its part of the network is islanded without synchronous generation (the `island_carriers` printed below), since inverter-based solar and wind farms disconnect when they lose the grid; (c) the generation left in its part of the network cannot cover the demand; or (d) the remaining lines would exceed their ratings.

Each failure is solved at peak and at average demand, which gives two quantities: `unmet_peak_mw`, the unmet demand in the worst hour, and `unmet_mwh_per_day`, the unmet energy per day of outage, which is the unmet demand at average demand over 24 hours. A restoration time in days turns the second into the energy not supplied. It is exact when whole areas lose supply, and too low for shortfalls that occur only near peak, as when a plant fails on a small island.

In [ ]:
import importlib
import json
import pathlib
import sys

for _candidate in (pathlib.Path.cwd(), *pathlib.Path.cwd().parents):
    if (_candidate / "_helpers.py").exists():
        sys.path.insert(0, str(_candidate))
        break
# _helpers finds the workflow's files and draws them.
import _helpers as h  # noqa: E402

importlib.reload(h)  # so edits to _helpers.py apply without restarting the kernel
if not h.found(h.CRITICALITY_DIR / "criticality.csv", h.RUN_CRITICALITY):
    raise RuntimeError("Make the criticality tables first, with the command above.")
summary, by_area = h.load_criticality()
metadata = json.loads((h.CRITICALITY_DIR / "metadata.json").read_text())
print("model data:                    ", h.MODEL_DATA)
print("max_radial_line_voltage_kv:    ", metadata["max_radial_line_voltage_kv"])
print("island_carriers:               ", metadata["island_carriers"])
print("unmet demand, intact network:   ", metadata["unmet_with_nothing_failed"])

## Demand by substation area

The first table gives each island's peak and average demand and its source (`demand_levels.csv`); the second gives each area's share of its island's demand (`service_weights_substations.csv`) and the resulting demand in MW. The rule `build_energy_demand` writes both files to `data/processed/energy/<model_data>/demand/inferred-provided-<region>/`. A substation's failure leaves the whole demand of its area unmet.

In [ ]:
display(h.demand_levels())
h.area_demand(summary).round({"share_of_island": 3, "peak_mw": 1, "average_mw": 1})

## Transmission line ratings

The map shows the rating the analysis gave each transmission line (`line_ratings.csv`), by its design voltage (colour) and number of circuits (width). A line designed for a higher voltage than it runs at is given a larger standard conductor, and so has a higher rating.

In [ ]:
# clip: None for the islands with transmission lines, or a (minx, miny, maxx, maxy) box in degrees
h.plot_line_ratings(clip=None);

## Unmet demand when each component fails, by cause

The analysis writes two tables: `criticality.csv` (`summary` here), with one row per component and its two quantities, and `criticality_by_area.csv` (`by_area`), with one row for each failure and each substation area left with unmet demand.

The first table below counts, for each island, the substations, lines and plants failed and the failures that leave demand unmet, next to the island's peak demand and plant capacity. Every component is failed in turn, plants included. A plant's failure removes only that plant, so it leaves demand unmet only when the island's other plants cannot cover the demand or the remaining lines reach their ratings: when `plant_capacity_mw` minus `largest_plant_mw` exceeds `peak_demand_mw`, no plant failure on the island falls short of generation.

The second table lists every component whose failure leaves demand unmet, with its two quantities, and splits the unmet demand at peak by the causes in the method: `failed_substation_area_peak_mw` is the demand of the failed substation's own area (a), zero when a line or plant fails, since lines and plants carry no demand; `areas_without_supply` and `areas_without_supply_peak_mw` are the other areas that lose all their supply, as when they are islanded without synchronous generation (b); and `areas_partly_supplied_peak_mw` is the unmet demand in areas that keep some supply (c or d). `line_limits_peak_mw` is the part due to line ratings (d), and shows whether any single failure overloads the remaining lines. Components not listed leave no demand unmet.

The third table lists, for each of those failures, every substation area with unmet demand: `criticality_by_area.csv` with the names of the failed component (`asset`) and the area (`area`), the area's status in the same three classes (`area_status`), and the share of the area's peak demand that is unmet (`unmet_share_of_peak`).

In [ ]:
h.failures_by_island(summary).round(1)

In [ ]:
unmet = h.unmet_by_cause(summary, by_area)
unmet.round(1)

In [ ]:
h.unmet_by_area(summary, by_area).round({"unmet_share_of_peak": 2, "unmet_mwh_per_day": 1, "unmet_peak_mw": 1})

## Criticality of transmission lines, substations and plants

The map colours each transmission line, substation and plant by the unmet demand when that component fails, in the quantity set by `QUANTITY`. Plants are drawn at their own sites. The power flow attaches each plant to its nearest substation without modelling the connection between them (the plant's own line or the distribution network), so a substation's failure also cuts off its plants. A line's failure leaves demand unmet only when it separates part of the network from enough synchronous generation or overloads the lines that remain.

In [ ]:
QUANTITY = "unmet_mwh_per_day"  # or "unmet_peak_mw"
# clip: None for the islands with transmission lines, or a (minx, miny, maxx, maxy) box in degrees
h.plot_criticality_map(summary, QUANTITY, clip=None);

## Distribution network without supply when one component fails

The map shows the failure of the component set by `ASSET`: the distribution lines in the areas that lose supply, coloured by the share of the area's peak demand that is unmet, and the failed component in red. The box at the top left gives the two quantities and the causes of the unmet demand at peak.

In [ ]:
# asset_id of any component in summary; unmet above lists those whose failure leaves demand unmet.
# The default is the component whose failure leaves the most demand without supply beyond its own area.
# Change iloc to select a different component based on its rank in areas_without_supply_peak_mw.
ASSET = unmet.sort_values("areas_without_supply_peak_mw", ascending=False, kind="stable")["asset_id"].iloc[3]
# clip: None for the component's island, or a (minx, miny, maxx, maxy) box in degrees
h.plot_failure(ASSET, summary=summary, by_area=by_area, clip=None);

## Unmet demand when each distribution line fails

The rule `build_energy_criticality_connectivity` fails every component again, from the connectivity of the network alone: with no line ratings and no power flow, it also covers the distribution lines. For substations, transmission lines and plants it gives the power flow's results wherever no line rating binds, which the first table checks: `largest_difference_peak_mw` is the largest difference between the two rules for one component.

Each distribution line is failed with the distribution network run in two ways. Run radially, each substation feeds its area along the shortest routes through the area's lines, and a failed line cuts off the nodes beyond it. After switching, a failed line cuts off demand only if no other route through the area's lines remains. The lines between areas stay open in both, so no demand moves to another area.

The second table counts, for each island, the distribution lines whose failure leaves at least 1 MW unmet at peak, and the third lists the lines whose failure leaves the most demand unmet, with their substation area. The largest values after switching are on the first road segment from a substation that the build joins to the roads at the end of a road: every route from that substation then starts on that segment, where a real substation has several feeders. The map colours each line by the unmet demand when it fails, in the quantity set by `DISTRIBUTION_QUANTITY`.

In [ ]:
if not h.found(h.CONNECTIVITY_DIR / "criticality.csv", h.RUN_CONNECTIVITY):
    raise RuntimeError("Make the connectivity tables first, with the command above.")
connectivity, connectivity_by_area = h.load_criticality(h.CONNECTIVITY_DIR)
h.compare_methods(summary, connectivity).round(1)

In [ ]:
display(h.distribution_line_counts(connectivity, threshold_mw=1.0).round(1))
h.largest_distribution_failures(connectivity, connectivity_by_area, count=10).round(1)

In [ ]:
DISTRIBUTION_QUANTITY = "unmet_peak_mw"  # or "unmet_after_switching_peak_mw", "unmet_mwh_per_day",
# "unmet_after_switching_mwh_per_day"
# clip: None for the island with the most distribution lines, or a (minx, miny, maxx, maxy) box in degrees
h.plot_distribution_criticality(connectivity, DISTRIBUTION_QUANTITY, clip=None);